# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [1]:
import sys
sys.path.append("..")
from common_config import llm_connect
model ="gpt-5.4-mini"

temperature = 0
max_tokens = 2086

llm = llm_connect(model=model)
# llm = llm_connect(model=model, temperature=temperature, max_tokens=max_tokens)

In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
LangGraph는 **LLM 애플리케이션을 “그래프(graph)” 형태로 설계하고 실행**할 수 있게 해주는 프레임워크입니다.  
LangChain 위에서 동작하며, 단순한 대화형 체인보다 더 복잡한 **상태 관리, 반복, 분기, 다중 에이전트 협업**을 쉽게 만들 수 있습니다.

## LangGraph가 필요한 이유
기존의 LLM 앱은 보통:
- 한 번 입력 → 한 번 출력
- 순차적인 체인(chain)

형태가 많습니다.  
하지만 실제 앱은 보통 더 복잡합니다:
- 사용자의 답변을 보고 **분기**
- 결과가 만족스럽지 않으면 **반복**
- 여러 에이전트가 서로 역할을 나눠 **협업**
- 중간 상태를 저장하고 **재개**
- 툴 호출 후 결과를 다시 판단

이런 흐름을 잘 다루기 위해 LangGraph는 **그래프 기반 실행 모델**을 제공합니다.

---

## LangGraph의 핵심 개념

### 1. Node(노드)
그래프의 각 작업 단위입니다.  
예를 들면:
- LLM 호출
- 검색
- 툴 실행
- 검증
- 메모리 업데이트

각 노드는 보통 입력 상태를 받아 새로운 상태를 반환합니다.

---

### 2. Edge(엣지)
노드 간의 연결입니다.  
어떤 노드 다음에 어떤 노드로 갈지 정의합니다.

- **일반 edge**: A → B처럼 고정된 흐름
- **conditional edge**: 조건에 따라 다른 노드로 분기

예:
- 답변이 충분히 좋으면 종료
- 아니면 재질문
- 도구가 필요하면 tool node로 이동

---

### 3. State(상태)
LangGraph의 매우 중요한 개념입니다.  
그래프 전체에서 공유되는 데이터 구조로, 보통 다음을 담습니다:
- 대화 기록
- 중간 추론 결과
- 툴 호출 결과
- 플래너의 판단
- 사용자 프로필

즉, **각 노드가 상태를 읽고 쓰며** 전체 흐름이 이어집니다.

---

### 4. Reducer / State Update
여러 노드가 상태를 갱신할 수

In [3]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 **LangChain의 확장 도구**로, **그래프 기반의 멀티 에이전트 워크플로우**를 구성할 수 있게 해줍니다.  
복잡한 AI 파이프라인을 구현할 때 **State, Node, Edge**라는 개념을 사용합니다.

- **State**: 작업의 현재 상태를 저장
- **Node**: 실행되는 각 단계나 에이전트
- **Edge**: 노드 간의 연결과 흐름

즉, LangGraph는 여러 에이전트와 단계들을 그래프 구조로 연결해 AI 워크플로우를 설계하는 방식입니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```